In [1]:
# 1. LIBRAIRIES ET CHARGEMENT DES DONNÉES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
import re


In [2]:

# Chargement des données
df = pd.read_csv("./X_train/clinical_train.csv")
df_eval = pd.read_csv("./X_test/clinical_test.csv")
maf_df = pd.read_csv("./X_train/molecular_train.csv")
maf_eval = pd.read_csv("./X_test/molecular_test.csv")
target_df = pd.read_csv("./target_train.csv")

# 2. NETTOYAGE ET EXPLORATION
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)
target_df = target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'])

df = df[df['ID'].isin(target_df['ID'])].reset_index(drop=True)
maf_df = maf_df[maf_df['ID'].isin(target_df['ID'])].reset_index(drop=True)

# Imputation avancée (KNN pour variables continues)
num_cols = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
cat_cols = ['CENTER', 'CYTOGENETICS']

imputer_num = KNNImputer(n_neighbors=5)
df[num_cols] = imputer_num.fit_transform(df[num_cols])
df_eval[num_cols] = imputer_num.transform(df_eval[num_cols])

# Imputation des catégorielles par le mode
for col in cat_cols:
    mode = df[col].mode()[0]
    df[col] = df[col].fillna(mode)
    df_eval[col] = df_eval[col].fillna(mode)

In [3]:
# 3. FEATURE ENGINEERING AVANCÉ

# 3.1. CYTOGÉNÉTIQUE : extraction fine + regroupement par risque
def extract_cytogenetic_features(df):
    cyto = df['CYTOGENETICS'].str.lower().fillna('')
    features = pd.DataFrame(index=df.index)
    features['cyto_normal'] = cyto.str.contains(r'46,xy|46,xx') & (cyto.str.len() < 10)
    features['cyto_complex'] = cyto.str.count(',') >= 3
    features['cyto_-5'] = cyto.str.contains('-5|monosomy 5')
    features['cyto_-7'] = cyto.str.contains('-7|monosomy 7')
    features['cyto_+8'] = cyto.str.contains('\+8|trisomy 8')
    features['cyto_del5q'] = cyto.str.contains('del\(5q\)|5q-')
    features['cyto_del7q'] = cyto.str.contains('del\(7q\)|7q-')
    features['cyto_inv16'] = cyto.str.contains('inv\(16\)')
    features['cyto_t821'] = cyto.str.contains('t\(8;21\)')
    features['cyto_t1517'] = cyto.str.contains('t\(15;17\)')
    features['cyto_risk'] = 1  # intermédiaire par défaut
    features.loc[features['cyto_normal'], 'cyto_risk'] = 0  # favorable
    features.loc[features['cyto_complex'] | features['cyto_-5'] | features['cyto_-7'], 'cyto_risk'] = 2  # défavorable
    return features.astype(int)

cyto_train = extract_cytogenetic_features(df)
cyto_eval = extract_cytogenetic_features(df_eval)

# 3.2. MOLECULAIRE : features avancées
def extract_molecular_features(maf, ids):
    key_genes = ['TP53', 'FLT3', 'NPM1', 'DNMT3A', 'IDH1', 'IDH2', 'ASXL1', 'RUNX1', 'TET2', 'SRSF2', 'SF3B1']
    features = pd.DataFrame({'ID': ids})
    maf['GENE'] = maf['GENE'].str.upper()
    maf['EFFECT'] = maf['EFFECT'].str.upper()
    maf['VAF'] = pd.to_numeric(maf['VAF'], errors='coerce')
    for gene in key_genes:
        features[f'mut_{gene}'] = features['ID'].map(lambda x: int(gene in maf[maf['ID'] == x]['GENE'].values))
    mut_count = maf.groupby('ID').size().reindex(ids, fill_value=0)
    features['mut_count'] = mut_count.values
    high_impact = maf[maf['EFFECT'].isin(['MODERATE', 'HIGH'])].groupby('ID').size().reindex(ids, fill_value=0)
    features['mut_high_impact'] = high_impact.values
    vaf_stats = maf.groupby('ID')['VAF'].agg(['mean', 'max', 'min', 'std']).reindex(ids)
    vaf_stats = vaf_stats.fillna(0)
    features = pd.concat([features, vaf_stats.reset_index(drop=True)], axis=1)
    return features

mol_train = extract_molecular_features(maf_df, df['ID'])
mol_eval = extract_molecular_features(maf_eval, df_eval['ID'])

# 3.3. CLINIQUE : ratios, binarisation, interactions
df['HB_low'] = (df['HB'] < 10).astype(int)
df['PLT_low'] = (df['PLT'] < 100).astype(int)
df['WBC_high'] = (df['WBC'] > 20).astype(int)
df['ANC_low'] = (df['ANC'] < 1).astype(int)
df['MONO_high'] = (df['MONOCYTES'] > 1).astype(int)
df['blast_ratio'] = df['BM_BLAST'] / (df['WBC'] + 1)
df['plt_hb_ratio'] = df['PLT'] / (df['HB'] + 1)

df_eval['HB_low'] = (df_eval['HB'] < 10).astype(int)
df_eval['PLT_low'] = (df_eval['PLT'] < 100).astype(int)
df_eval['WBC_high'] = (df_eval['WBC'] > 20).astype(int)
df_eval['ANC_low'] = (df_eval['ANC'] < 1).astype(int)
df_eval['MONO_high'] = (df_eval['MONOCYTES'] > 1).astype(int)
df_eval['blast_ratio'] = df_eval['BM_BLAST'] / (df_eval['WBC'] + 1)
df_eval['plt_hb_ratio'] = df_eval['PLT'] / (df_eval['HB'] + 1)

# 3.4. Encodage des variables catégorielles
df['CENTER'] = LabelEncoder().fit_transform(df['CENTER'])
df_eval['CENTER'] = LabelEncoder().fit_transform(df_eval['CENTER'])

# 3.5. Fusion de toutes les features
X_train = pd.concat([df.reset_index(drop=True), cyto_train.reset_index(drop=True), mol_train.reset_index(drop=True)], axis=1)
X_eval = pd.concat([df_eval.reset_index(drop=True), cyto_eval.reset_index(drop=True), mol_eval.reset_index(drop=True)], axis=1)

# 3.6. Suppression des colonnes inutiles
drop_cols = ['ID', 'CYTOGENETICS']
X_train = X_train.drop(columns=drop_cols)
X_eval = X_eval.drop(columns=drop_cols)

# 4. STANDARDISATION
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled = scaler.transform(X_eval)

# 5. FORMAT SURVIE
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

<>:11: SyntaxWarning: invalid escape sequence '\+'
<>:12: SyntaxWarning: invalid escape sequence '\('
<>:13: SyntaxWarning: invalid escape sequence '\('
<>:14: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
<>:16: SyntaxWarning: invalid escape sequence '\('
<>:11: SyntaxWarning: invalid escape sequence '\+'
<>:12: SyntaxWarning: invalid escape sequence '\('
<>:13: SyntaxWarning: invalid escape sequence '\('
<>:14: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
<>:16: SyntaxWarning: invalid escape sequence '\('
/var/folders/90/srl1fpkn6cv5r29n0_ct_tz80000gn/T/ipykernel_1119/1283395302.py:11: SyntaxWarning: invalid escape sequence '\+'
  features['cyto_+8'] = cyto.str.contains('\+8|trisomy 8')
/var/folders/90/srl1fpkn6cv5r29n0_ct_tz80000gn/T/ipykernel_1119/1283395302.py:12: SyntaxWarning: invalid escape sequence '\('
  features['cyto_del5q'] = cyto.str.contains('del\(5q\)|5q-')
/var/folders/90

In [4]:

# 6. MODELE : RANDOM SURVIVAL FOREST + OPTIMISATION
rsf = RandomSurvivalForest(
    n_estimators=500,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    max_depth=12,
    n_jobs=-1,
    random_state=42
)
rsf.fit(X_train_scaled, y)

# 7. EVALUATION
rsf_pred_train = rsf.predict(X_train_scaled)
rsf_c_index_train = concordance_index_ipcw(y, y, rsf_pred_train, tau=7)[0]
print(f"RSF Concordance Index IPCW (train): {rsf_c_index_train:.4f}")

# 8. PREDICTION FINALE
prediction_on_test_set = rsf.predict(X_eval_scaled)
submission_rsf = pd.DataFrame({'ID': df_eval['ID'], 'risk_score': prediction_on_test_set})
submission_rsf.to_csv('./rsf_submission.csv', index=False)
print("Fichier de soumission généré : rsf_submission.csv")


RSF Concordance Index IPCW (train): 0.8246
Fichier de soumission généré : rsf_submission.csv
